EinsteinPy test codes

In [ ]:
import sympy
from sympy import symbols, sin, cos, sinh, Matrix
from sympy import init_session
from einsteinpy.symbolic import EinsteinTensor, MetricTensor
# sympy.init_session # initialize all sympy contents, symbols, including init_printing
sympy.init_printing() # initializethe best printing format in the current environent


In [3]:
syms = sympy.symbols("t x y z") 
A, W = sympy.symbols("A W") 
t, x, y, z = syms

M = ([[-1,0,0,0], [0,1,0,0],[0,0,1,A*sin(W*(t-x))], [0,0,A*sin(W*(t-x)),1]])
M

In [5]:
metric = MetricTensor(M, syms)
metric.tensor() # Input - metric tensor

⎡-1  0         0                 0        ⎤
⎢                                         ⎥
⎢0   1         0                 0        ⎥
⎢                                         ⎥
⎢0   0         1          A⋅sin(W⋅(t - x))⎥
⎢                                         ⎥
⎣0   0  A⋅sin(W⋅(t - x))         1        ⎦

In [6]:
einst = EinsteinTensor.from_metric(metric) # Calculating the Einstein Tensor (with both indices covariant)

einst.tensor() # Output:  Einstein Tensor 

⎡      ⎛                                              ⎛   2    2               ↪
⎢ 2  2 ⎜  ⎛ 2    2               ⎞                    ⎝3⋅A ⋅sin (W⋅(t - x)) -  ↪
⎢A ⋅W ⋅⎜- ⎝A ⋅sin (W⋅(t - x)) - 1⎠⋅cos(2⋅W⋅(t - x)) + ──────────────────────── ↪
⎢      ⎝                                                                  2    ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                           2                  ↪
⎢                                   ⎛ 2    2               ⎞                   ↪
⎢                                   ⎝A ⋅sin (W⋅(t - x)) - 1⎠                   ↪
⎢                                                                              ↪
⎢      ⎛⎛     2    2               ⎞    2                                      ↪
⎢ 2  2 ⎜⎝- 3⋅A ⋅sin (W⋅(t - x)) + 1⎠⋅cos (W⋅(t - x))   ⎛ 2    2                ↪
⎢A ⋅W ⋅⎜──────────────────────────────────────────── + ⎝A ⋅sin (W⋅(t - x)) - 1 ↪
⎢      ⎝                    

Anti-de Sitter spacetime Metric

In [ ]:
syms = sympy.symbols("t chi theta phi") 
syms
t, ch, th, ph = syms
m = sympy.diag(-1, cos(t) ** 2, cos(t) ** 2 * sinh(ch) ** 2, cos(t) ** 2 * sinh(ch) ** 2 * sin(th) ** 2).tolist()
metric = MetricTensor(m, syms)
metric.tensor() # Input - metric tensor
einst = EinsteinTensor.from_metric(metric) # Calculating the Einstein Tensor (with both indices covariant)
einst.tensor() # Output:  Einstein Tensor 

Solving the 2 coupled differential equations for the neutron stars — the "mass continuity equation and the "Tolman-Oppenheimer-Volkoff (TOV) equation" require (all in geometrized units):
1. Specify an equation of state (EOS) p(rho) relating pressure p and energy density rho.
2. Integrate numerically from the center (r = 0) to the surface (p = 0), using appropriate boundary conditions.

3. Equations to Solve<br>
$dm/dr = 4pi*r^2*rho$ (Mass continuity)<br>
$dp/dr = -(rho + p)(m + 4pi*r^3*p)/{r(r - 2m)}$ (TOV equation)

4. Boundary Conditions
At the center r = 0, m(0) = 0 (no mass enclosed at r = 0,
p(0) = p_c (central pressure, chosen based on the star's properties).
At the surface r = R: p(R) = 0 (pressure drops to zero at the surface).

5. Equation of State (EOS)
A simple but physically relevant EOS is the "polytropic equation of state":
p = K*rho^Gamma, where: K is a constant, Gamma is the adiabatic index (e.g., Gamma = 2 for neutron stars).
For a more realistic model, tabulated EOS data (e.g., SLy, APR) can be used.

6. Numerical Integration (Pseudocode) 
We use the Runge-Kutta 4th-order (RK4) method to integrate the equations outward from r = 0:
Example : for a neutron star with:
p_c = 10^35 Pa, rh_c = 5 * 10^17 kg/m^3, Gamma = 2, K = 1 * 10^5
You would obtain:
Mass M approx = 1.4 solar masses,
Radius R approx = 10 km.

Iteration 0 - 3 trial runs

In [ ]:
import numpy as np
from sympy import *

# Iteration 0
n = 0
print(f"Iteration = {n}")
Gamma = 2
K = 1*10**5
r0 = 0.01
rho_0 = (5*10**17)*(7.42592*10**(-28)) # convert into geometrized unit
print(f"rho = {rho_0}")
print(f"r = {r0}")
m0 = 4/3 * np.pi * r0**3 * rho_0
print("m = {:3e}".format(m0))
print("r - 2m ={:3e}".format(r0 - 2*m0))
p0 = K * rho_0 ** Gamma
print("p = {:3e}".format(p0))
dp0_dr = - (rho_0 + p0) * (m0 + 4 * np.pi * r0**3 * p0) / (r0 * (r0 - 2 * m0) )
print("dp/dr ={:3e}".format(dp0_dr))

# Iteration 1
n += 1
print(f"Iteration = {n}")
r1 = r0 + 1
p1 = p0 + dp0_dr * 1
rho_1 = (p1 / K )**(1/Gamma)
print(f"rho = {rho_1}")
print(f"r = {r1}")
m1 = 4/3 * np.pi * r1**3 * rho_1
print("m = {:3e}".format(m1))
print("r - 2m ={:3e}".format(r1- 2*m1))
print("p = {:3e}".format(p1))
dp1_dr = - (rho_1 + p1) * (m1 + 4 * np.pi * r1**3 * p1) / (r1 * (r1 - 2 * m1) )
print("dp/dr ={:3e}".format(dp1_dr))

# Iteration 2
n += 1
print(f"Iteration = {n}")
r2 = r1 + 1
p2 = p1 + dp1_dr * 1
rho_2 = (p2 / K )**(1/Gamma)
print(f"rho = {rho_2}")
print(f"r = {r2}")
m2 = 4/3 * np.pi * r2**3 * rho_2
print("m = {:3e}".format(m2))
print("r - 2m ={:3e}".format(r2- 2*m2))
print("p = {:3e}".format(p2))
dp2_dr = - (rho_2 + p2) * (m2 + 4 * np.pi * r2**3 * p2) / (r2 * (r2 - 2 * m2) )
print("dp/dr ={:3e}".format(dp2_dr))

# Iteration 3
n += 1
print(f"Iteration = {n}")
r3 = r2 + 1
p3 = p2 + dp2_dr * 1
rho_3 = (p3 / K )**(1/Gamma)
print(f"rho = {rho_3}")
print(f"r = {r3}")
m3 = 4/3 * np.pi * r3**3 * rho_3
print("m = {:3e}".format(m3))
print("r - 2m ={:3e}".format(r3- 2*m3))
print("p = {:3e}".format(p3))
dp3_dr = - (rho_3 + p3) * (m3 + 4 * np.pi * r3**3 * p3) / (r3 * (r3 - 2 * m3) )
print("dp/dr ={:3e}".format(dp3_dr))
dp3_dr

Integration loop

In [5]:
import numpy as np
from sympy import *

def TOV_solver(p_c, rho_c, Gamma, K, r_max, dr):
    # Initialize
    n = 0
    r = 0.01
    m = 4/3 * np.pi * r**3 * rho_c
    # p = K * rho_c ** Gamma
    p = p_c
    rho = rho_c
    dp_dr = - (rho + p) * (m + 4 * np.pi * r**3 * p) / (r * (r - 2 * m) )
    print(f"\nIteration = {n}")
    print(f"rho = {rho_c}")
    print(f"r = {r}")
    print("m = {:3e}".format(m))
    print("r - 2m ={:3e}".format(r - 2*m))
    print("p = {:3e}".format(p))
    print("dp/dr ={:3e}".format(dp_dr))
    
    # Arrays to store results
    r_list = [r]
    m_list = [m]
    p_list = [p]
    rho_list = [rho]
    
    # Integration loop

    while r < r_max and p > 0:
        # Compute derivatives
        n += 1
        r += dr
        p += dp_dr * dr
        rho = (p / K)**(1 / Gamma)
        dm_dr = 4 * np.pi * r**2 * rho
        m += dm_dr * dr
        dp_dr = - (rho + p) * (m + 4 * np.pi * r**3 * p) / (r * (r - 2 * m) )
        if n < 5 or n > 950:
            print(f"\nIteration = {n}")
            print(f"rho = {rho}")
            print(f"r = {r}")
            print("m = {:3e}".format(m))
            print("r - 2m ={:3e}".format(r- 2*m))
            print("p = {:3e}".format(p))
            print("dp/dr ={:3e}".format(dp_dr))
       
        # Store results
        r_list.append(r)
        m_list.append(m)
        p_list.append(p)
        rho_list.append(rho)

    print("r_list, m_list, p_list, rho_list:")

    return 
    # return r_list, m_list, p_list, rho_list

TOV_solver(p_c = (10**35)*(8.2624*10**(-45)), rho_c = (5*10**17)*(7.42592*10**(-28)), Gamma = 2, K = 1*10**5, r_max=10, dr=0.01)



Iteration = 0
rho = 3.7129599999999997e-10
r = 0.01
m = 1.555281e-15
r - 2m =1.000000e-02
p = 8.262400e-10
dp/dr =-1.429633e-19

Iteration = 1
rho = 9.089774474642208e-08
r = 0.02
m = 4.570574e-12
r - 2m =2.000000e-02
p = 8.262400e-10
dp/dr =-1.067125e-15

Iteration = 2
rho = 9.089774415942989e-08
r = 0.03
m = 1.485087e-11
r - 2m =3.000000e-02
p = 8.262400e-10
dp/dr =-1.542105e-15

Iteration = 3
rho = 9.08977433111664e-08
r = 0.04
m = 3.312694e-11
r - 2m =4.000000e-02
p = 8.262400e-10
dp/dr =-1.937179e-15

Iteration = 4
rho = 9.089774224558512e-08
r = 0.05
m = 6.168331e-11
r - 2m =5.000000e-02
p = 8.262400e-10
dp/dr =-2.310753e-15

Iteration = 951
rho = 9.080816305033654e-08
r = 9.519999999999841
m = 3.288359e-04
r - 2m =9.519342e+00
p = 8.246122e-10
dp/dr =-3.415360e-13

Iteration = 952
rho = 9.080797499655607e-08
r = 9.529999999999841
m = 3.298723e-04
r - 2m =9.529340e+00
p = 8.246088e-10
dp/dr =-3.418931e-13

Iteration = 953
rho = 9.080778674576733e-08
r = 9.539999999999841
m = 3.3

Integration loop

In [ ]:
import numpy as np
from sympy import *

def TOV_solver(p_c, rho_c, Gamma, K, r_max, dr):
    # Initialize
    n = 0
    print(f"\nIteration = {n}")
    r = 0.01
    m = 4/3 * np.pi * r**3 * rho_c
    print(f"rho = {rho_c}")
    print(f"r = {r}")
    print("m = {:3e}".format(m))
    print("r - 2m ={:3e}".format(r - 2*m))
    
    # p = K * rho_c ** Gamma
    p = p_c
    print("p = {:3e}".format(p))
    rho = rho_c
    dp_dr = - (rho + p) * (m + 4 * np.pi * r**3 * p) / (r * (r - 2 * m) )
    print("dp/dr ={:3e}".format(dp_dr))
    
    # Arrays to store results
    r_list = [r]
    m_list = [m]
    p_list = [p]
    rho_list = [rho]
    
    # Integration loop
    

    while r < r_max and p > 0:
        # Compute derivatives
        n += 1
        print(f"\nIteration = {n}")
        r += dr
        p += dp_dr * dr
        rho = (p / K)**(1 / Gamma)
        print(f"rho = {rho}")
        print(f"r = {r}")
        dm_dr = 4 * np.pi * r**2 * rho
        m += dm_dr * dr
        print("m = {:3e}".format(m))
        print("r - 2m ={:3e}".format(r- 2*m))
        print("p = {:3e}".format(p))
    
        dp_dr = - (rho + p) * (m + 4 * np.pi * r**3 * p) / (r * (r - 2 * m) )
        print("dp/dr ={:3e}".format(dp_dr))
       
        # Store results
        r_list.append(r)
        m_list.append(m)
        p_list.append(p)
        rho_list.append(rho)

    print("r_list, m_list, p_list, rho_list:")
    r_list
    return 
    # return r_list, m_list, p_list, rho_list

TOV_solver(p_c = (10**35)*(8.2624*10**(-45)), rho_c = (5*10**17)*(7.42592*10**(-28)), Gamma = 2, K = 1*10**5, r_max=10**4, dr=0.01)

    


5. Results
The integration stops when p drops to zero, defining the star's radius R.
The total gravitational mass is M = m(R).
The density rho(r) and pressure p(r) profile describe the star's internal structure.

Example : for a neutron star with:
p_c = 10^35 Pa, rho_c = 5 * 10^17 kg/m^3, Gamma = 2, K = 1 * 10^5
You would obtain:
Mass M approx = 1.4 solar masses,
Radius R approx = 10 km.

Key Notes
Relativistic effects: The TOV equation deviates from Newtonian gravity when m/r is large (e.g., near neutron star cores).
Stability: Solutions are valid only if M and R satisfy stability criteria.